In [8]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

# =========================
# THE 3 BANKS (PROJECT VERSION)
# =========================
BANKS = {
    "Commercial Bank of Ethiopia (CBE)": "com.combanketh.mobilebanking",
    "Bank of Abyssinia (BOA)": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# =========================
# STEP 1: GET APP METADATA (FOR ALL BANKS)
# =========================
for bank_name, app_id in BANKS.items():

    app_info = app(
        app_id,
        lang='en',
        country='et'
    )

    print("=" * 50)
    print(f"{bank_name} App Info")
    print("=" * 50)
    print(f"App Title   : {app_info['title']}")
    print(f"Current Score: {app_info['score']}")
    print(f"Total Ratings: {app_info['ratings']:,}")
    print(f"Total Reviews: {app_info['reviews']:,}")
    print(f"Installs     : {app_info['installs']}")

# =========================
# STEP 2: SCRAPE REVIEWS
# =========================
all_results = []

for bank_name, app_id in BANKS.items():

    print(f"\nScraping reviews for {bank_name}...")

    result, continuation_token = reviews(
        app_id,
        lang='en',
        country='et',
        sort=Sort.NEWEST,
        count=500,
        filter_score_with=None
    )

    print(f"Collected {len(result)} raw reviews")

    print("Keys in a single review:")
    print(list(result[0].keys()))

    print("\nFirst raw review (sample):")
    for key, value in result[0].items():
        print(f"  {key}: {value}")
        break

    # Step 3: Extract only the columns we need
    raw_data = []

    for r in result:
        raw_data.append({
            'review_id': r.get('reviewId', ''),
            'review'   : r.get('content', ''),
            'rating'   : r.get('score', None),
            'date'     : r.get('at', None),
            'bank'     : bank_name,
            'source'   : 'Google Play'
        })

    all_results.extend(raw_data)

# =========================
# BUILD DATAFRAME
# =========================
df_raw = pd.DataFrame(all_results)

print(f"Shape: {df_raw.shape}")
df_raw.head()

# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

# Sample dates
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

# =========================
# DATA QUALITY AUDIT
# =========================
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# Missing values
print("\nProblem 1: Missing Values")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

# Duplicates
print("Problem 2: Duplicates")

exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

# Date format
print("Problem 3: Date Format")
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Target format: YYYY-MM-DD")

# =========================
# CLEANING
# =========================
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

before = len(df)

# Drop missing values
df = df.dropna(subset=['review', 'rating'])

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")

before = len(df)

# Remove duplicates
df = df.drop_duplicates(subset=['review_id'])

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")

# =========================
# DATE NORMALIZATION
# =========================
print("Before normalization:")
print(df['date'].head(3).to_string())

df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("After normalization:")
print(df['date'].head(3).to_string())

print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# =========================
# TEXT CLEANING
# =========================
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

df['review'] = df['review'].apply(clean_text)

before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"Removed empty reviews: {removed}")

# =========================
# RATING VALIDATION
# =========================
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")

# =========================
# FINAL DATASET
# =========================
df_clean = df[['review', 'rating', 'date', 'bank', 'source']]

df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

# =========================
# SAVE TO CSV
# =========================
import os
os.makedirs('data/processed', exist_ok=True)

output_path = 'data/processed/clean_reviews.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

# =========================
# PREPROCESSING REPORT
# =========================
print("=" * 55)
print("PREPROCESSING REPORT — BANKS")
print("=" * 55)

original_count = len(df_raw)
final_count = len(df_clean)
removed_total = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\nRaw reviews collected  : {original_count}")
print(f"Reviews after cleaning : {final_count}")
print(f"Reviews removed        : {removed_total}")
print(f"Data retention rate    : {retention_rate:.1f}%")

print("\nBank distribution:")
print(df_clean['bank'].value_counts())

print("\nRating distribution:")
print(df_clean['rating'].value_counts().sort_index(ascending=False))

print("\nText length stats:")
lengths = df_clean['review'].str.len()
print(f"Min: {lengths.min()}")
print(f"Median: {lengths.median():.0f}")
print(f"Max: {lengths.max()}")

print("\nColumns in final CSV:")
for col in df_clean.columns:
    print("-", col)


Libraries loaded successfully!
Commercial Bank of Ethiopia (CBE) App Info
App Title   : Commercial Bank of Ethiopia
Current Score: 4.2834234
Total Ratings: 48,263
Total Reviews: 9,305
Installs     : 5,000,000+
Bank of Abyssinia (BOA) App Info
App Title   : BoA Mobile
Current Score: 4.392136
Total Ratings: 9,202
Total Reviews: 1,457
Installs     : 1,000,000+
Dashen Bank App Info
App Title   : Dashen Bank
Current Score: 4.25222
Total Ratings: 5,592
Total Reviews: 1,017
Installs     : 1,000,000+

Scraping reviews for Commercial Bank of Ethiopia (CBE)...
Collected 500 raw reviews
Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 56185597-d29b-4a60-a0fb-6783638230a7

Scraping reviews for Bank of Abyssinia (BOA)...
Collected 500 raw reviews
Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUp